In [1]:
import sys
import pandas as pd
sys.path.insert(0, "..")
from backtest.db import connect, query_df

with connect() as conn:
    ModelData = query_df(conn, "select * from backtest.models")

ModelData.shape

(21562, 51)

In [2]:
ModelData['period_end'] = pd.to_datetime(ModelData['period_end'])
ModelData['rebalance_date'] = pd.to_datetime(ModelData['rebalance_date'])

# Growth is a property of each statement, so compute it on one row per
# (ticker, period_end), not on ModelData, which repeats a statement for ~12
# rebalance dates. temp_models also includes statements no rebalance date used.
with connect() as conn:
    statements = query_df(conn, """
        select ticker, period_end, revenue
        from backtest.temp_models
        where revenue > 0
    """)

statements['period_end'] = pd.to_datetime(statements['period_end'])
statements = statements.sort_values(['ticker', 'period_end'])
by_ticker = statements.groupby('ticker')

In [3]:
def cagr(k):
    """Revenue CAGR from the statement k years back: (Rev_T / Rev_{T-k})^(1/years) - 1."""
    years = (statements['period_end'] - by_ticker['period_end'].shift(k)).dt.days / 365.25
    return (statements['revenue'] / by_ticker['revenue'].shift(k)) ** (1 / years) - 1

statements['g0_raw'] = cagr(2).fillna(cagr(1))   # 2-year CAGR, else 1-year
statements['g0'] = statements['g0_raw'].clip(-0.10, 0.25)

#add the growth figs from the statements into the ModelData
ModelData = ModelData.drop(columns=['g0_raw', 'g0'], errors='ignore').merge(
    statements[['ticker', 'period_end', 'g0_raw', 'g0']],
    on=['ticker', 'period_end'], how='left')

In [4]:
statements[statements['ticker'] == 'AAPL'] #[['ticker', 'period_end', 'rebalance_date', 'g0']]

,ticker,period_end,revenue,g0_raw,g0
4,AAPL,2023-09-30,3.832850e+11,NaN,NaN
5,AAPL,2024-09-30,3.910350e+11,0.020178,0.020178
6,AAPL,2025-09-30,4.161610e+11,0.041976,0.041976
7,AAPL,2026-09-30,4.788340e+11,0.106661,0.106661


In [5]:
statements.shape

(2011, 5)

In [6]:
import numpy as np
from scipy.optimize import brentq

N = 5 # forecast horizon in years. growth fades linearly from g0 to g_L over this period
R_MAX = 1.0 # upper bracket for implied return, above 100% is treated as unsolvable

# Fixed assumption for (long-run growth / terminal growth) for all firms, rather than per firm: G_L = 2.5% nominal.
G_L = 0.025 
ModelData['g_L'] = np.minimum(G_L, ModelData['riskfree']) #Upper bound (long-run growth rate can't be higher than rf forever)
ModelData['g0_missing'] = ModelData['g0'].isna()
ModelData['g0_used'] = ModelData['g0'].fillna(ModelData['g_L'])

#Enterprise value at the rebalance date (T = 0) which is market cap ('equity') + ex-lease net debt.
ModelData['ev'] = ModelData['equity'] + ModelData['netdebt']

# Shared by every model below
def fade(x0, x_L):
    """Linear path x_k = x0 + (x_L - x0) k/N for k = 1...N, reaching x_L in year N."""
    return x0 + (x_L - x0) * np.arange(1, N + 1) / N

def grow(x0, g0, g_L):
    """x0 compounded along the faded growth path: x_t = x0 * (1 + g_1)...(1 + g_t)."""
    return x0 * np.cumprod(1 + fade(g0, g_L))

def pv(r, flows, g_L):
    """PV of flows for years 1...N, plus
    Gordon terminal value flows_N (1 + g_L) / (r - g_L) discounted from year N."""
    disc = (1 + r) ** np.arange(1, N + 1)
    terminal = flows[-1] * (1 + g_L) / (r - g_L)
    return (flows / disc).sum() + terminal / disc[-1]

def solve_r(value, price, g_L):
    """Solve for r (implied return) that sets value(r) = price, or NaN if r > R_MAX"""
    f = lambda r: value(r) - price
    lo = g_L + 1e-6 #approaches +inf as r approaches g_L
    if not (f(lo) > 0 > f(R_MAX)): # no root in (g_L, R_MAX], also rejects NaN inputs
        return np.nan
    return brentq(f, lo, R_MAX) #brentq solver bounded by R_MAX

def apply_rows(fn, cols):
    """fn(*row) for every row of ModelData[cols]."""
    return [fn(*row) for row in ModelData[cols].itertuples(index=False, name=None)]

def implied_r(cf0, g0, g_L, price):
    """Implied return for a cash flow cf0 growing along the fade.
    Used for FCFF (against EV) and shareholder yield (against market cap)."""
    if not (cf0 > 0 and price > 0):
        return np.nan
    flows = grow(cf0, g0, g_L)
    return solve_r(lambda r: pv(r, flows, g_L), price, g_L)

ModelData['r_fcff'] = apply_rows(implied_r, ['fcff', 'g0_used', 'g_L', 'ev'])


In [7]:
ModelData[ModelData['ticker'] == 'AAPL'][['ticker', 'period_end', 'rebalance_date', 'g0', 'r_fcff']]

,ticker,period_end,rebalance_date,g0,r_fcff
3742,AAPL,2023-09-30,2023-11-30,NaN,0.056190
4234,AAPL,2023-09-30,2023-12-29,NaN,0.055777
4726,AAPL,2023-09-30,2024-01-31,NaN,0.057283
5218,AAPL,2023-09-30,2024-02-29,NaN,0.057926
5710,AAPL,2023-09-30,2024-03-28,NaN,0.059928
6204,AAPL,2023-09-30,2024-04-30,NaN,0.060160
6698,AAPL,2023-09-30,2024-05-31,NaN,0.056200
7192,AAPL,2023-09-30,2024-06-28,NaN,0.053608
7686,AAPL,2023-09-30,2024-07-31,NaN,0.052242
8180,AAPL,2023-09-30,2024-08-30,NaN,0.051427


In [8]:
MIN_PEERS = 5 # a sector median from fewer firms than this is left empty (same rule as the dashboard's comps)

# Shared by the RIM, revenue and comps models
def sector_median(values):
    """Median of values across each (rebalance_date, sector). Only uses statements available at that date."""
    groups = values.groupby([ModelData['rebalance_date'], ModelData['sector']])
    return groups.transform('median').where(groups.transform('count') >= MIN_PEERS)

# Reverse RIM: market cap = B0 + PV(residual income), RI_t = (ROE_t - r) B_{t-1}
# ROE fades from the firm's current ROE to its sector median; book grows by retained earnings (clean surplus).
# Terminal RI grows at the same g_L as the FCFF model.
book = ModelData['bookequity'].where(ModelData['bookequity'] > 0)
ModelData['roe'] = ModelData['netincome'] / book
ModelData['roe_sector'] = sector_median(ModelData['roe'])
ModelData['payout'] = (-ModelData['dividendspaid'] / ModelData['netincome']).clip(0, 1).fillna(0) # dividends only, loss years retain nothing

def implied_r_rim(b0, roe0, roe_L, payout, g_L, price):
    """Implied cost of equity that sets B0 + PV(residual income) = market cap"""
    if not (b0 > 0 and price > 0):
        return np.nan
    roe = fade(roe0, roe_L)
    b_prev = b0 * np.cumprod(np.r_[1, 1 + roe[:-1] * (1 - payout)]) # B_{t-1} for t = 1...N
    return solve_r(lambda r: b0 + pv(r, (roe - r) * b_prev, g_L), price, g_L)

ModelData['r_rim'] = apply_rows(implied_r_rim, ['bookequity', 'roe', 'roe_sector', 'payout', 'g_L', 'equity'])


In [9]:
# Reverse revenue: EV = PV(revenue x FCFF margin). Revenue grows along the same fade as FCFF,
# and the margin fades from the firm's current FCFF margin to its sector median, so loss-makers still get a value.
revenue = ModelData['revenue'].where(ModelData['revenue'] > 0)
ModelData['margin'] = ModelData['fcff'] / revenue
ModelData['margin_sector'] = sector_median(ModelData['margin'])

def implied_r_rev(rev0, g0, g_L, m0, m_L, ev):
    """Implied return that sets PV(revenue x margin) = EV"""
    if not (rev0 > 0 and m_L > 0 and ev > 0): # a negative terminal margin can't be solved
        return np.nan
    flows = grow(rev0, g0, g_L) * fade(m0, m_L)
    return solve_r(lambda r: pv(r, flows, g_L), ev, g_L)

ModelData['r_rev'] = apply_rows(implied_r_rev, ['revenue', 'g0_used', 'g_L', 'margin', 'margin_sector', 'ev'])

In [10]:
# Shareholder yield: dividends + net buybacks against market cap, growing along the same fade as FCFF.
# Both are stored negative when cash leaves the firm, so the payout is minus their sum.
ModelData['shareholder_payout'] = -(ModelData['dividendspaid'].fillna(0) + ModelData['netstockissuance'].fillna(0))

ModelData['r_shy'] = apply_rows(implied_r, ['shareholder_payout', 'g0_used', 'g_L', 'equity'])

In [11]:
# COMPS: not an implied return, so it gives an upside instead: the equity value implied by the
# sector's median EV/EBITDA and P/E (averaged, like the dashboard), over market cap, minus 1.
ebitda = ModelData['ebitda'].where(ModelData['ebitda'] > 0)
earnings = ModelData['netincome'].where(ModelData['netincome'] > 0)
ev_ebitda = (ModelData['ev'] / ebitda).where(ModelData['ev'] > 0)
pe = (ModelData['equity'] / earnings).where(ModelData['equity'] > 0)

equity_from_ev = sector_median(ev_ebitda) * ebitda - ModelData['netdebt']
equity_from_pe = sector_median(pe) * earnings
implied_equity = pd.concat([equity_from_ev.where(equity_from_ev > 0), equity_from_pe], axis=1).mean(axis=1)

ModelData['comps_upside'] = implied_equity / ModelData['equity'] - 1

In [12]:
model_cols = ['r_fcff', 'r_rim', 'r_rev', 'r_shy', 'comps_upside']
dev = ModelData[ModelData['split'] == 'development']
dev[model_cols].describe(percentiles=[.05, .25, .5, .75, .95]).T

,count,mean,std,min,5%,25%,50%,75%,95%,max
r_fcff,8218.0,0.070399,0.049059,0.025146,0.031338,0.047825,0.059930,0.077252,0.139685,0.992217
r_rim,10110.0,0.081154,0.060203,0.025052,0.033343,0.051199,0.067791,0.094837,0.163513,0.992624
r_rev,8739.0,0.078186,0.070758,0.026033,0.034943,0.044210,0.057211,0.080855,0.202385,0.951402
r_shy,9933.0,0.070119,0.036909,0.025004,0.030813,0.046459,0.062412,0.081008,0.140394,0.356704
comps_upside,10478.0,0.247976,1.600268,-0.999886,-0.692297,-0.295956,0.005456,0.450651,1.771681,36.611184
